# Day 17 · 规模化训练：DeepSpeed / FSDP

**配套讲义**: [`days/day-17.md`](../days/day-17.md) ｜ **本地可跑，不需要 GPU**

搞清 ZeRO-1/2/3 与 FSDP 的取舍，写出两份能直接用的 DeepSpeed 配置；并回答「为什么 VLM 训练里视觉塔通常冻结」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 校验两份配置

In [ ]:
import json
from pathlib import Path

for name in ("deepspeed_zero2.json", "deepspeed_zero3.json"):
    p = Path("../configs") / name
    cfg = json.loads(p.read_text())
    print("=" * 60)
    print(name)
    print("  stage       :", cfg.get("zero_optimization", {}).get("stage"))
    print("  offload     :", cfg.get("zero_optimization", {}).get("offload_optimizer", {}).get("device", "none"))
    print("  bf16        :", cfg.get("bf16", {}).get("enabled"))
    print("  gather_16bit:", cfg.get("zero_optimization", {}).get("stage3_gather_16bit_weights_on_model_save"))

## 2. 显存账：切了之后各卡还剩下多少

ZeRO-2 把**梯度和优化器状态**按卡数摊分，ZeRO-3 连权重也摊。
手算一下：3 张卡开 ZeRO-3，单卡静态显存是多少？和 Day 13 的估算对一下。

In [ ]:
P = 3.0e9
GB = 1024 ** 3
n_gpu = 3

for stage, w_share, g_share, o_share in [
    ("ZeRO-1", 1.0, 1.0, 1 / n_gpu),
    ("ZeRO-2", 1.0, 1 / n_gpu, 1 / n_gpu),
    ("ZeRO-3", 1 / n_gpu, 1 / n_gpu, 1 / n_gpu),
]:
    w = P * 2 / GB * w_share        # bf16 全参权重
    g = P * 2 / GB * g_share
    o = P * 8 / GB * o_share
    print(f"{stage}: 权重 {w:6.1f} + 梯度 {g:6.1f} + 优化器 {o:6.1f} = {w+g+o:6.1f} GB/卡（3 卡）")

## 3. 落笔：为什么视觉塔冻结

把三个理由写下来（提示：参数量占比 / 预训练已充分 / 梯度不稳定对底层特征的破坏）。

In [ ]:
why_freeze = """
1.
2.
3.
"""
print(why_freeze)

## 验收清单

- [ ] 两份配置 JSON 合法，且每项都有注释说明为什么这么写
- [ ] 能说清 ZeRO-2 和 ZeRO-3 的取舍（省显存 vs 通信开销）
- [ ] 能说出 VLM 训练里视觉塔冻结的理由，以及「冻结 ≠ 不占显存」
- [ ] 知道 `stage3_gather_16bit_weights_on_model_save` 是 ZeRO-3 存模型时的必踩坑

**卡住了？** 回看 [`days/day-17.md`](../days/day-17.md) 第五节「容易踩的坑」。

> **明天**：`days/day-18.md` —— 合并 LoRA、20 条并排抽检，W3 收官